# 02 - Preprocessing: Missing Values & Outliers

Two things are handled here:
1. Zero values in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI` are treated as missing and imputed with the per-class (`Outcome`) median.
2. Outliers are detected (IQR method) and capped (winsorized).

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

from src.eda import grab_col_names, check_missing_value
from src.preprocessing import zeros_to_missing, check_outlier, replace_with_thresholds

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv("../data/diabetes.csv")
df_copy = df.copy()

## I. Missing values

A mean BMI of 12 is the lower limit compatible with human survival (James et al., 1988). Let's check for implausible rows first.

In [3]:
df_copy[df_copy['BMI'] < 12]

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
9,8,125,96,0,0,0.0,0.232,54,1
49,7,105,0,0,0,0.0,0.305,24,0
60,2,84,0,0,0,0.0,0.304,21,0
81,2,74,0,0,0,0.0,0.102,22,0
145,0,102,75,23,0,0.0,0.572,21,0
371,0,118,64,23,89,0.0,1.731,21,0
426,0,94,0,0,0,0.0,0.256,25,0
494,3,80,0,0,0,0.0,0.174,22,0
522,6,114,0,0,0,0.0,0.189,26,0
684,5,136,82,0,0,0.0,0.640,69,0


In [4]:
cat_cols, num_cols, cat_but_car = grab_col_names(df_copy, print_results=False)
df_copy = zeros_to_missing(df_copy)
check_missing_value(df_copy)

Glucose: 5 zero values converted to NaN
BloodPressure: 35 zero values converted to NaN
SkinThickness: 227 zero values converted to NaN
Insulin: 374 zero values converted to NaN
BMI: 11 zero values converted to NaN
Empty DataFrame
Columns: [n_miss, ratio]
Index: []


## II. Outlier detection & handling

Using the IQR method (10th/90th percentile, 1.5x IQR whiskers).

In [5]:
for col in num_cols:
    print(col, check_outlier(df_copy, col))

Pregnancies False
Glucose False
BloodPressure False
SkinThickness True
Insulin True
BMI False
DiabetesPedigreeFunction True
Age False


In [6]:
for col in num_cols:
    replace_with_thresholds(df_copy, col)

for col in num_cols:
    print(col, check_outlier(df_copy, col))

Pregnancies False
Glucose False
BloodPressure False
SkinThickness False
Insulin False
BMI False
DiabetesPedigreeFunction False
Age False


## Save the cleaned dataset

Persisted so downstream notebooks can pick it up without recomputing this step.

In [7]:
df_copy.to_csv("../data/diabetes_cleaned.csv", index=False)
df_copy.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6.0,148.0,72.0,35.0,169.5,33.6,0.627,50.0,1
1,1.0,85.0,66.0,29.0,102.5,26.6,0.351,31.0,0
2,8.0,183.0,64.0,32.0,169.5,23.3,0.672,32.0,1
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0,0
4,0.0,137.0,40.0,35.0,168.0,43.1,1.949,33.0,1
